# Can a literal &ge;10 filter reproduce their table?

**The objection.** LLMGPR states plainly: *"Following [20, 28], users and POIs with less than 10
interactions are removed."* Our recovered build uses a user cut of 60, which contradicts the
paper. Their `#users` (7,507) and `#check-ins per user` (162.80) each independently imply ~63–66
on the **raw three-city dump**, so &ge;10 cannot be the cut *relative to that baseline*.

**The reconciliation to test.** &ge;10 can still be literally true if their starting pool was
already activity-selected — then their "10" and our "60" are the same cut measured against
different baselines. This notebook asks whether a *principled* pre-selection exists that makes
the literal &ge;10 land on their table.

**The prime candidate is not a guess.** Yang's §5 release ships two check-in files, and
`dataset_WWW_Checkins_anonymized.txt` — the headline one most people download — we already
proved is exactly *"raw check-ins of the friendship users"* (output3). It is therefore already
socially and activity selected, and its global mean is **199.5 check-ins per user** against
LLMGPR's 162.80 — a factor of 0.82, where the raw dump is off by 3.3&times;. The window is narrow
and does not need tuning:

```
14,401   three-city users in the filtered file           (>=10 on a GLOBAL basis keeps ~all)
 7,507   <-- THEIRS sits between the two
 5,116   those clearing >=10 measured IN-REGION          (rule E, output3)
```

So the question is only *which basis* the 10 is measured on. That is what we sweep.

**The measurement we are missing** is each user's **global** check-in count — every sweep so far
counted only check-ins inside the three cities. One pass over each check-in file fixes that, and
it also re-opens §1.8: the "arXiv NYC is unattainable" bound assumed in-region counting, and does
not apply if check-ins are counted over whole histories.

Needs the zip (~20 s download). Attach **output3** and **output5** for the cached extract and
POI coordinates.

## 0. Setup and the cached three-city extract

In [ ]:
import os, re, gc, math, zipfile, subprocess, itertools
import pandas as pd, numpy as np

WORK = "/kaggle/working"; os.makedirs(WORK, exist_ok=True)
CHUNK = 2_000_000
TARGET = dict(users=7_507, pois=80_962, cats=436, ck=1_214_631)
TARGET_NYC = dict(users=6_078, pois=63_445, cats=436, ck=923_856)
CITY_CENTRE = {"New York": (40.707864, -73.905237), "Chicago": (41.826546, -87.641298),
               "Los Angeles": (34.000002, -118.250001)}

def find(pat, roots=("/kaggle/input", WORK)):
    hits = []
    for root in roots:
        if not os.path.isdir(root): continue
        for dp, _, fns in os.walk(root):
            if "__MACOSX" in dp: continue
            for fn in fns:
                if re.search(pat, fn, re.I) and not fn.startswith("._"):
                    hits.append(os.path.join(dp, fn))
    return sorted(hits)

def load(pat, what, **kw):
    p = find(pat); assert p, f"{what} not found -- attach output3 and output5"
    print("loading", p[0])
    return pd.read_parquet(p[0]) if p[0].endswith(".parquet") else pd.read_csv(p[0], **kw)

ck   = load(r"llmgpr_checkins_A\.(parquet|csv)$", "3-city check-ins", dtype=str)
pois = load(r"llmgpr_pois_xy\.(parquet|csv)$",    "POI coordinates", dtype={"venue_id": str})
pois["lat"] = pd.to_numeric(pois["lat"], errors="coerce")
pois["lon"] = pd.to_numeric(pois["lon"], errors="coerce")
pois = pois.dropna(subset=["lat", "lon"]).drop_duplicates("venue_id").reset_index(drop=True)
print(f"{len(ck):,} check-ins | {ck['user_id'].nunique():,} users | {len(pois):,} venues")

## 1. Fetch the two check-in files and the friendship edges

In [ ]:
NEED = dict(raw_ck=r"raw_Checkins.*\.txt$", filt_ck=r"WWW_Checkins.*\.txt$",
            f_old=r"friendship_old.*\.txt$")
P = {k: find(v) for k, v in NEED.items()}
if not all(P.values()):
    ZIP = f"{WORK}/dataset_WWW2019.zip"
    url = ("https://drive.usercontent.google.com/download?"
           "id=1PNk3zY8NjLcDiAbzjABzY5FiPAFHq6T8&export=download&confirm=t")
    print("fetching the zip")
    assert subprocess.run(f'curl -L --fail --retry 3 -o "{ZIP}" "{url}"', shell=True).returncode == 0
    with zipfile.ZipFile(ZIP) as z:
        for n in z.namelist():
            if "__MACOSX" in n or n.endswith("/"): continue
            if re.search(r"(raw_Checkins|WWW_Checkins|friendship_old)", n):
                z.extract(n, WORK); print("  extracted", n, flush=True)
    os.remove(ZIP); P = {k: find(v) for k, v in NEED.items()}
P = {k: v[0] for k, v in P.items()}
for k, v in P.items(): print(f"{k:<9}: {v}  ({os.path.getsize(v) / 1024**3:.2f} GB)")

## 2. Per-user check-in counts on every basis

Four bases per user, three of them new:

- `n_region` — check-ins inside the three-city bounding boxes (what every sweep so far used)
- `n_R10` — check-ins inside the 10 km catalogue region
- `n_global_raw` — the user's **entire** history in `raw_Checkins`, worldwide
- `n_global_filt` — the user's entire history in the filtered (friendship-users) file

Membership of the filtered file is itself the pre-selection we are testing, so we record it.

In [ ]:
def per_user_counts(path, label, restrict=None):
    """Total check-ins per user over a whole file. restrict: set of user_ids to keep."""
    acc, seen = None, 0
    for chx in pd.read_csv(path, sep="\t", header=None,
                           names=["user_id", "venue_id", "utc_time", "tz"],
                           dtype={"user_id": str}, usecols=[0], on_bad_lines="skip",
                           chunksize=CHUNK):
        seen += len(chx)
        s = chx["user_id"]
        if restrict is not None: s = s[s.isin(restrict)]
        vc = s.value_counts()
        acc = vc if acc is None else acc.add(vc, fill_value=0)
        print(f"\r{label}: scanned {seen:,}", end="", flush=True)
    print(f"\n{label}: {int(acc.sum()):,} check-ins over {len(acc):,} users")
    return acc.astype("int64")

region_users = set(ck["user_id"].unique())
g_raw  = per_user_counts(P["raw_ck"],  "raw     ", restrict=region_users)
g_filt = per_user_counts(P["filt_ck"], "filtered")          # all of it: defines membership
print(f"\nfiltered file covers {len(g_filt):,} users globally")
print(f"  of our {len(region_users):,} three-city users, "
      f"{len(region_users & set(g_filt.index)):,} appear in it")

In [ ]:
# distance per venue to its own city centre -> the R-km catalogue and n_R10
def haversine_km(lat, lon, lat0, lon0):
    R = 6371.0088
    p, p0 = np.radians(lat), math.radians(lat0)
    dp, dl = p - p0, np.radians(lon - lon0)
    return 2 * R * np.arcsin(np.sqrt(np.sin(dp / 2) ** 2 +
                                     np.cos(p) * math.cos(p0) * np.sin(dl / 2) ** 2))

v2city = dict(zip(ck["venue_id"], ck["city"]))
pois = pois.assign(city=pois["venue_id"].map(v2city))
vkm = np.full(len(pois), np.inf)
for c, (la, lo) in CITY_CENTRE.items():
    m = (pois["city"] == c).to_numpy()
    if m.any():
        vkm[m] = haversine_km(pois["lat"].to_numpy()[m], pois["lon"].to_numpy()[m], la, lo)
pois = pois.assign(km=vkm)
ck = ck.assign(km=ck["venue_id"].map(dict(zip(pois["venue_id"], pois["km"]))))

edges = pd.read_csv(P["f_old"], sep="\t", header=None, names=["u", "v"], dtype=str)
social = set(edges["u"]) | set(edges["v"])

U = pd.DataFrame({"user_id": sorted(region_users)})
U["n_region"]      = U["user_id"].map(ck["user_id"].value_counts()).fillna(0).astype("int64")
U["n_R10"]         = U["user_id"].map(ck[ck["km"] <= 10]["user_id"].value_counts()).fillna(0).astype("int64")
U["n_global_raw"]  = U["user_id"].map(g_raw).fillna(0).astype("int64")
U["n_global_filt"] = U["user_id"].map(g_filt).fillna(0).astype("int64")
U["in_filt"]       = U["user_id"].isin(g_filt.index)
U["has_edge"]      = U["user_id"].isin(social)

print(U[["n_region", "n_R10", "n_global_raw", "n_global_filt"]].describe().loc[["mean", "50%", "max"]].round(1))
print(f"\nin filtered file: {U['in_filt'].sum():,}   with a friendship_old edge: {U['has_edge'].sum():,}")
print("\nmean check-ins per user, by pool and basis:")
for pool, mask in (("all three-city", np.ones(len(U), bool)),
                   ("in filtered file", U["in_filt"].to_numpy()),
                   ("has social edge", U["has_edge"].to_numpy())):
    sub = U[mask]
    print(f"  {pool:<18} n={len(sub):>7,}  region {sub['n_region'].mean():>7.1f}  "
          f"global_raw {sub['n_global_raw'].mean():>7.1f}  global_filt {sub['n_global_filt'].mean():>7.1f}")

## 3. The sweep — with the threshold pinned at 10

Pool &times; threshold basis &times; reporting basis &times; catalogue radius. `#POIs` stays the region
catalogue, which §1.4 settled. The question is whether **Tu = 10** reaches their table under any
principled combination; the wider threshold list is only there for context.

In [ ]:
POOLS = {"all":       np.ones(len(U), bool),
         "filt_file": U["in_filt"].to_numpy(),
         "social":    U["has_edge"].to_numpy()}
BASES = ["n_region", "n_R10", "n_global_raw", "n_global_filt"]
RADII = [8, 10, 12, 999]
TUS   = [10, 15, 20, 30, 40, 60]

cat_by_R = {R: (int((pois["km"] <= R).sum()),
                int(pois.loc[pois["km"] <= R, "category"].nunique())) for R in RADII}
print("catalogue by radius:", {R: v[0] for R, v in cat_by_R.items()})

def match4(got, t=TARGET):
    return float(np.mean([min(got[k], t[k]) / max(got[k], t[k]) for k in ("users", "pois", "cats", "ck")]))

rows = []
for pname, pmask in POOLS.items():
    for basis, report, R, Tu in itertools.product(BASES, BASES, RADII, TUS):
        sel = pmask & (U[basis].to_numpy() >= Tu) & (U[report].to_numpy() > 0)
        if not sel.any(): continue
        npoi, ncat = cat_by_R[R]
        got = dict(users=int(sel.sum()), pois=npoi, cats=ncat,
                   ck=int(U.loc[sel, report].sum()))
        rows.append((match4(got), pname, basis, report, R, Tu, got))
rows.sort(key=lambda r: -r[0])

def show(rs, title, n=12):
    print(f"\n### {title}")
    hdr = (f"{'match':>7}{'pool':>11}{'threshold on':>15}{'counted on':>15}{'R':>5}{'Tu':>4}"
           f"{'users':>9}{'POIs':>9}{'cats':>6}{'check-ins':>12}{'ck/user':>9}")
    print(hdr); print("-" * len(hdr))
    for m, pn, b, rp, R, Tu, g in rs[:n]:
        print(f"{m:>7.3f}{pn:>11}{b.replace('n_',''):>15}{rp.replace('n_',''):>15}"
              f"{(R if R != 999 else 'all'):>5}{Tu:>4}{g['users']:>9,}{g['pois']:>9,}"
              f"{g['cats']:>6}{g['ck']:>12,}{g['ck'] / g['users']:>9.1f}")
    print("-" * len(hdr))
    print(f"{'TARGET':>7}{'':>11}{'':>15}{'':>15}{'':>5}{'':>4}{TARGET['users']:>9,}"
          f"{TARGET['pois']:>9,}{TARGET['cats']:>6}{TARGET['ck']:>12,}"
          f"{TARGET['ck'] / TARGET['users']:>9.1f}")

show([r for r in rows if r[5] == 10], "THRESHOLD PINNED AT 10 -- the paper's stated rule")
show(rows, "any threshold, for reference")

## 4. Verdict

In [ ]:
best10 = max((r for r in rows if r[5] == 10), key=lambda r: r[0])
bestany = rows[0]
m, pn, b, rp, R, Tu, g = best10
print(f"best with the literal Tu = 10:  {m:.3f}   pool={pn}, threshold on {b.replace('n_','')}, "
      f"counted on {rp.replace('n_','')}, R={R if R != 999 else 'all'}")
print(f"{'column':<20}{'ours':>12}{'theirs':>12}{'ratio':>9}")
print("-" * 53)
for k, lab in (("users", "users"), ("pois", "POIs"), ("cats", "categories"), ("ck", "check-ins")):
    print(f"{lab:<20}{g[k]:>12,}{TARGET[k]:>12,}{g[k] / TARGET[k]:>9.2f}x")
print(f"{'check-ins per user':<20}{g['ck'] / g['users']:>12.1f}"
      f"{TARGET['ck'] / TARGET['users']:>12.1f}"
      f"{(g['ck'] / g['users']) / (TARGET['ck'] / TARGET['users']):>9.2f}x")

print(f"\nbest at any threshold:          {bestany[0]:.3f}  (Tu={bestany[5]}, pool={bestany[1]}, "
      f"threshold on {bestany[2].replace('n_','')}, counted on {bestany[3].replace('n_','')})")
print(f"previous best (raw dump, Tu=60, 10 km catalogue): 0.978\n")

if m >= 0.95:
    print("=> THE PAPER'S RULE IS VINDICATED. A principled pre-selection plus the literal >=10")
    print("   reproduces their table. Build on this: it follows their stated method exactly and")
    print("   no longer needs a footnote explaining a threshold of 60.")
elif m >= 0.90:
    print("=> CLOSE. >=10 works on this pool to within a few percent. Prefer it over Tu=60 on")
    print("   grounds of fidelity to their stated method, and report the residual honestly.")
elif m >= 0.85:
    print("=> PARTIAL. >=10 is viable on this pool but fits worse than Tu=60 on the raw dump.")
    print("   Report both: '>=10 as stated, on the filtered release' and 'Tu=60 on the raw dump'.")
else:
    print("=> NO. No principled pre-selection makes the literal >=10 reproduce their table.")
    print("   The >=10 sentence cannot be squared with their own #users and #check-ins-per-user.")
    print("   Report Tu=60 with the two-independent-estimates argument as justification.")

print("\n--- side result: does global counting rescue the arXiv NYC table? (S1.8) ---")
nyc_v = set(pois.loc[pois["city"].eq("New York"), "venue_id"])
nyc_u = ck[ck["venue_id"].isin(nyc_v)]["user_id"].value_counts()
UN = U.set_index("user_id")
cand = UN.reindex(nyc_u.index)
for basis in ("n_region", "n_global_raw", "n_global_filt"):
    tot = cand[basis].fillna(0).to_numpy()
    order = np.sort(tot)[::-1]
    k = TARGET_NYC["users"]
    if k <= len(order):
        share = order[:k].sum() / max(tot.sum(), 1)
        need = TARGET_NYC["ck"] / max(tot.sum(), 1)
        verdict = "ATTAINABLE" if need <= share else "impossible"
        print(f"  counted on {basis.replace('n_',''):<12} top {k:,} hold {share:.1%} of the pool"
              f"  | their table needs {need:.1%}  -> {verdict}")
    else:
        print(f"  counted on {basis.replace('n_',''):<12} only {len(order):,} NYC users in the "
              f"pool, fewer than their {k:,} -- cannot be their region at all")
print("  If a basis puts their required share BELOW the top-k share, S1.8's impossibility")
print("  was an artifact of in-region counting and that table is back in play.")